# Capítulo 6 — Teoria de Carteiras e Asset Pricing

Companion em Python inspirado na sequência conceitual do Volume I de *Market Risk Analysis: Quantitative Methods in Finance*, de Carol Alexander.

> Objetivo: implementar os conceitos matemáticos e financeiros, não reproduzir o texto do livro.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import log, exp, sqrt

from quantfinance.performance import omega_ratio, sharpe_ratio, sortino_ratio
from quantfinance.portfolio import gmv_weights

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [ ]:
from scipy.optimize import minimize
import statsmodels.api as sm

## I.6.2 — Utilidade, aversão ao risco e equivalente certo

In [ ]:
def exponential_utility(w, a=3.0):
    return -np.exp(-a*w)

wealth = np.linspace(0, 2, 300)
plt.figure(figsize=(7,4))
plt.plot(wealth, exponential_utility(wealth))
plt.title("Utilidade exponencial")
plt.show()

In [ ]:
# CE aproximado sob critério média-variância
mu = 0.10
sigma = 0.20
risk_aversion = 4

CE = mu - 0.5*risk_aversion*sigma**2
print("Equivalente certo aproximado:", CE)

## I.6.3.1 — Diversificação

In [ ]:
sigma1, sigma2 = 0.20, 0.40
w = 0.25
rhos = np.linspace(-1,1,200)

vols = np.sqrt(
    w**2*sigma1**2 +
    (1-w)**2*sigma2**2 +
    2*rhos*w*(1-w)*sigma1*sigma2
)

plt.figure(figsize=(8,4))
plt.plot(rhos, vols)
plt.xlabel("Correlação")
plt.ylabel("Volatilidade da carteira")
plt.title("Efeito da correlação sobre o risco")
plt.show()

## I.6.3.2 — Carteira global de mínima variância

In [ ]:
V = np.array([
    [0.15**2,  0.5*0.15*0.20, -0.7*0.15*0.40],
    [0.5*0.15*0.20, 0.20**2, -0.4*0.20*0.40],
    [-0.7*0.15*0.40, -0.4*0.20*0.40, 0.40**2]
])

ones = np.ones(3)
Vinv = np.linalg.inv(V)
w_gmv = Vinv @ ones / (ones @ Vinv @ ones)

print("Pesos GMV:", w_gmv)
print("Vol GMV:", np.sqrt(w_gmv @ V @ w_gmv))

## I.6.3.3–5 — Markowitz e fronteira eficiente

In [ ]:
mu = np.array([0.05, 0.06, 0.00])

def min_var_for_target(target):
    n = len(mu)
    def obj(w): return w @ V @ w
    cons = [
        {"type":"eq","fun":lambda w: np.sum(w)-1},
        {"type":"eq","fun":lambda w: w@mu-target}
    ]
    res = minimize(obj, x0=np.repeat(1/n,n), constraints=cons)
    return res.x, np.sqrt(res.fun)

targets = np.linspace(-0.01,0.08,100)
frontier = np.array([min_var_for_target(t)[1] for t in targets])

plt.figure(figsize=(8,5))
plt.plot(frontier, targets)
plt.xlabel("Volatilidade")
plt.ylabel("Retorno esperado")
plt.title("Fronteira média-variância")
plt.show()

## I.6.4.1 — Capital Market Line

In [ ]:
Rf = 0.05
mu_m = 0.10
sigma_m = 0.20

sharpe_m = (mu_m-Rf)/sigma_m
sigmas = np.linspace(0,0.50,100)
cml = Rf + sharpe_m*sigmas

plt.figure(figsize=(8,4))
plt.plot(sigmas, cml)
plt.scatter([sigma_m],[mu_m])
plt.xlabel("Volatilidade")
plt.ylabel("Retorno esperado")
plt.title("Capital Market Line")
plt.show()

print("Sharpe do mercado:", sharpe_m)

## I.6.4.2–3 — CAPM, beta e SML

In [ ]:
Rf = 0.05
Rm = 0.11
beta = 1.5

required = Rf + beta*(Rm-Rf)
print("Retorno requerido CAPM:", required)

In [ ]:
betas = np.linspace(-0.2, 2.0, 200)
sml = Rf + betas*(Rm-Rf)

plt.figure(figsize=(8,4))
plt.plot(betas, sml)
plt.xlabel("Beta")
plt.ylabel("Retorno esperado")
plt.title("Security Market Line")
plt.show()

## I.6.4.4 — Teste empírico do CAPM

In [ ]:
rng = np.random.default_rng(123)
n = 1000
rm_excess = rng.normal(0.0003,0.012,n)
alpha_true = 0.0001
beta_true = 1.2
ri_excess = alpha_true + beta_true*rm_excess + rng.normal(0,0.008,n)

capm = sm.OLS(ri_excess, sm.add_constant(rm_excess)).fit()
print(capm.summary())

## I.6.5 — Métricas de performance ajustadas ao risco

In [ ]:
r = rng.normal(0.0005,0.012,2000)
print("Sharpe :", sharpe_ratio(r))
print("Sortino:", sortino_ratio(r))
print("Omega  :", omega_ratio(r))

### Sharpe ajustado por autocorrelação: demonstração

In [ ]:
# Simulação AR(1) para mostrar que sqrt(tempo) pode ser inadequado
rho = 0.3
eps = rng.normal(0,0.01,3000)
r = np.zeros_like(eps)

for t in range(1,len(r)):
    r[t] = rho*r[t-1] + eps[t]

daily_sharpe = r.mean()/r.std(ddof=1)
naive_annual = daily_sharpe*np.sqrt(252)

# variância anual empírica por blocos de 252 dias
blocks = r[:(len(r)//252)*252].reshape(-1,252).sum(axis=1)
empirical_annual = blocks.mean()/blocks.std(ddof=1)

print("Sharpe anualizado ingênuo:", naive_annual)
print("Sharpe em blocos anuais   :", empirical_annual)

## Projeto final sugerido
Monte uma carteira com dados reais e produza:
- retornos simples e log;
- matriz de covariância/correlação;
- PCA;
- beta CAPM de cada ativo;
- carteira GMV;
- fronteira eficiente;
- Sharpe, Sortino e Omega;
- comparação com um benchmark.